In [ ]:
import torch
from transformers import PeAudioVideoModel, PeAudioVideoProcessor,AutoProcessor, AutoModel
import time
import numpy as np
from pathlib import Path
import glob
import os
import re
import torch
from natsort import natsorted  


# Base paths
BASE_DIR = Path("/home/amin/Research/Representation/Movie/data/Setareh/Data")
OUTPUT_PATH = Path ("/home/amin/Research/Representation/Movie/outputs/model_embeddings/pe-av-small-16-frame")
single_modality_output_path = Path (OUTPUT_PATH /"pe-av-small-16-frame-single-modality")
joint_modality_output_path = Path (OUTPUT_PATH /"pe-av-small-16-frame-joint-modality")

BATCH_SIZE = 16  # Can increase since 2s clips are small
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device is {DEVICE}")

device is cuda


In [2]:
model_id = "facebook/pe-av-small-16-frame"
import os
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"  # 5 minutes

model = PeAudioVideoModel.from_pretrained(model_id).to(DEVICE)
processor = PeAudioVideoProcessor.from_pretrained(model_id)
print("Model & Processor Ready")

Loading weights:   0%|          | 0/859 [00:00<?, ?it/s]

Model & Processor Ready


In [3]:
def extract_part_number(filename):
    """
    Extract part number from filename like 'video_part_001.mp4' or 'audio_part_042.wav'
    Returns the number as int, or None if not found
    """
    match = re.search(r'part[_-]?(\d+)', filename.stem, re.IGNORECASE)
    if match:
        return int(match.group(1))
    return None

# Gather all video/audio chunk pairs across all runs
all_videos = []
all_audios = []

for run_idx in range(1, 19):  # Video1 through Video18
    vid_chunk_dir = BASE_DIR / f"Video{run_idx}" / f"Video{run_idx}_chunks"
    aud_chunk_dir = BASE_DIR / f"Audio{run_idx}" / f"Audio{run_idx}_chunks"
    
    if not vid_chunk_dir.exists() or not aud_chunk_dir.exists():
        print(f"Warning: Missing dir for video {run_idx}")
        continue
    
    vids = natsorted(vid_chunk_dir.glob("*.mp4"))
    auds = natsorted(aud_chunk_dir.glob("*.wav"))
    if len(vids) == 0 or len(auds) == 0:
        print(f"Warning: No chunks found for video {run_idx}")
        continue\
    
    # Match by part number
    vid_parts = {}
    for v in vids:
        part_num = extract_part_number(v)
        if part_num is not None:
            vid_parts[part_num] = v
    
    aud_parts = {}
    for a in auds:
        part_num = extract_part_number(a)
        if part_num is not None:
            aud_parts[part_num] = a
    
    # Find common part numbers
    common_parts = sorted(set(vid_parts.keys()) & set(aud_parts.keys()))
    
    for part_num in common_parts:
        all_videos.append(vid_parts[part_num])
        all_audios.append(aud_parts[part_num])
    
    print(f"Video {run_idx}: {len(common_parts)} paired chunks (video: {len(vids)}, audio: {len(auds)})")

paired = list(zip(all_videos, all_audios))
print(f"\n{'='*60}")
print(f"Total: {len(paired)} paired 2s segments")
print(f"{'='*60}\n")

if len(paired) == 0:
    print("ERROR: No paired segments found!")
    exit(1)

Video 1: 106 paired chunks (video: 106, audio: 106)
Video 2: 110 paired chunks (video: 110, audio: 110)
Video 3: 93 paired chunks (video: 93, audio: 93)
Video 4: 31 paired chunks (video: 31, audio: 31)
Video 5: 41 paired chunks (video: 41, audio: 41)
Video 6: 113 paired chunks (video: 113, audio: 113)
Video 7: 129 paired chunks (video: 129, audio: 129)
Video 8: 124 paired chunks (video: 124, audio: 124)
Video 9: 41 paired chunks (video: 41, audio: 41)
Video 10: 90 paired chunks (video: 90, audio: 90)
Video 11: 92 paired chunks (video: 92, audio: 92)
Video 12: 101 paired chunks (video: 101, audio: 101)
Video 13: 70 paired chunks (video: 70, audio: 70)
Video 14: 41 paired chunks (video: 41, audio: 41)
Video 15: 116 paired chunks (video: 116, audio: 116)
Video 16: 115 paired chunks (video: 115, audio: 115)
Video 17: 127 paired chunks (video: 127, audio: 127)
Video 18: 41 paired chunks (video: 41, audio: 41)

Total: 1581 paired 2s segments



# Extract joint embeddings

In [ ]:
# Extract embeddings
video_out, audio_out, av_out = [], [], []
clip_names = []
vid_ids = []

with torch.inference_mode():
    for i in range(0, len(paired), BATCH_SIZE):
        batch_pairs = paired[i:i + BATCH_SIZE]
        v_batch = [str(v) for v, a in batch_pairs]
        a_batch = [str(a) for v, a in batch_pairs]
        
        inputs = processor(videos=v_batch, audio=a_batch, return_tensors="pt").to(DEVICE)
        
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(**inputs)
        
        video_out.append(outputs.video_embeds.float().cpu())
        audio_out.append(outputs.audio_embeds.float().cpu())
        av_out.append(outputs.audio_video_embeds.float().cpu())
        
        for v, a in batch_pairs:
            clip_names.append(v.stem)
            vid_ids.append(v.parent.parent.name)
        
        if (i // BATCH_SIZE) % 50 == 0 or i == 0:
            print(f"Processed {i + len(batch_pairs)}/{len(paired)}")

video_joint_embeds = torch.cat(video_out, dim=0)
audio_joint_embeds = torch.cat(audio_out, dim=0)
av_embeds = torch.cat(av_out, dim=0)

print(f"\nDone!")
print(f"Video embeddings: {video_joint_embeds.shape}")
print(f"Audio embeddings: {audio_joint_embeds.shape}")
print(f"AV embeddings:    {av_embeds.shape}")

In [ ]:
np.save(Path(joint_modality_output_path / 'peav_small_16-frame_joint_av.npy'),av_embeds.numpy())
np.save(Path(joint_modality_output_path / 'peav_small_16-frame_joint_a.npy'),audio_joint_embeds.numpy())
np.save(Path(joint_modality_output_path / 'peav_small_16-frame_joint_v.npy'),video_joint_embeds.numpy())
print(f"joint embedding npy files saved to {joint_modality_output_path}")

In [ ]:
torch.save({
    "video": video_joint_embeds,
    "audio": audio_joint_embeds,
    "av": av_embeds,
    "clip_names": clip_names,
    "vid_ids": vid_ids,},
    joint_modality_output_path.with_suffix('.pt'))

print(f"Joint modality embeddings .pt saved to {OUTPUT_PATH.with_suffix('.pt')}")

In [ ]:
import numpy as np

np.savez(
    joint_modality_output_path.with_suffix('.npz'),
    video=video_joint_embeds.numpy(),
    audio=audio_joint_embeds.numpy(),
    av=av_embeds.numpy(),
    clip_names=np.array(clip_names),
    vid_ids=np.array(vid_ids),
)
print(f"joint modality .npz saved to {OUTPUT_PATH.with_suffix('.npz')}")

# Extract single embeddings

In [15]:
# Extract pre-fusion embeddings (single modality)
video_out_pure, audio_out_pure = [], []
clip_names = []
vid_ids = []

with torch.inference_mode():
    for i in range(0, len(paired), BATCH_SIZE):
        batch_pairs = paired[i:i + BATCH_SIZE]
        v_batch = [str(v) for v, a in batch_pairs]
        a_batch = [str(a) for v, a in batch_pairs]
        
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            # Process inputs
            inputs_v = processor(videos=v_batch, return_tensors="pt").to(DEVICE)
            inputs_a = processor(audio=a_batch, return_tensors="pt").to(DEVICE)
            
            # Use the correct methods
            video_only_embeds = model.video_model.get_video_features(
                pixel_values_videos=inputs_v['pixel_values_videos'],
                padding_mask_videos=inputs_v.get('padding_mask_videos', None)
            )
            
            audio_only_embeds = model.audio_model.get_audio_embeds(
                input_values=inputs_a['input_values'],
                padding_mask=inputs_a.get('padding_mask', None)
            )
        
        video_out_pure.append(video_only_embeds.float().cpu())
        audio_out_pure.append(audio_only_embeds.float().cpu())
        
        for v, a in batch_pairs:
            clip_names.append(v.stem)
            vid_ids.append(v.parent.parent.name)
        
        if (i // BATCH_SIZE) % 50 == 0 or i == 0:
            print(f"Processed {i + len(batch_pairs)}/{len(paired)}")

video_only_embeds = torch.cat(video_out_pure, dim=0)
audio_only_embeds = torch.cat(audio_out_pure, dim=0)

print(f"\nSingle Modality Embeddings Done!")
print(f"Video embeddings: {video_only_embeds.shape}")
print(f"Audio embeddings: {audio_only_embeds.shape}")

Processed 16/1581
Processed 816/1581

Single Modality Embeddings Done!
Video embeddings: torch.Size([1581, 1024])
Audio embeddings: torch.Size([1581, 1024])


In [ ]:
np.savez(
    single_modality_output_path.with_suffix('.npz'),
    video=video_only_embeds.numpy(),
    audio=audio_only_embeds.numpy(),
    clip_names=np.array(clip_names),
    vid_ids=np.array(vid_ids),
)
print(f"single modality embedding npz files saved to {single_modality_output_path.with_suffix('.npz')}")

single modality embedding npz files saved to /home/amin/Research/Representation/Movie/outputs/model_embeddings/pe-av-small/pe-av-small-16-frame-single-modality/pe-av-small-16-frame-single-modality.npz
